# AmicoScript — Google Colab Bridge

Run this notebook to use Google Colab's free GPU as your transcription backend. Your audio files, tags, and library stay safely on your local machine — only the heavy AI processing is offloaded to Colab.

---

## Step-by-step setup

### 1 — Select a GPU runtime
In the menu bar click **Runtime › Change runtime type**, set **Hardware accelerator** to **T4 GPU**, then click **Save**.

### 2 — Run the install cell
Run *Cell 1* below (click ▶ or press `Shift+Enter`). This clones the repo and installs all dependencies. It takes 2–4 minutes on a fresh runtime.

### 3 — Get a free ngrok token
ngrok creates a public HTTPS tunnel into this Colab session so AmicoScript can reach it.

1. Go to **[https://dashboard.ngrok.com/signup](https://dashboard.ngrok.com/signup)** and create a free account (GitHub/Google login works).
2. After signing in, open **[https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)**.
3. Copy the token string shown on that page.

### 4 — Paste your token and start the server
In *Cell 2*, replace `YOUR_NGROK_TOKEN_HERE` with the token you just copied, then run it.

After a few seconds you will see:
```
👉  Copy this URL into AmicoScript › Settings › Colab Bridge URL:
    https://xxxx-xxxx.ngrok-free.app
```

### 5 — Connect AmicoScript
1. Open AmicoScript on your computer.
2. Go to **Settings** (gear icon) → **Colab Bridge URL**.
3. Paste the `ngrok-free.app` URL and save.

> **Keep this tab open** while using AmicoScript. Closing it or letting Colab disconnect will drop the tunnel.

In [ ]:
# Cell 1 — Clone repo and install dependencies (~2–4 min on a fresh runtime)
import os

REPO_ROOT = "/content/amico-script"

if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/sim186/amico-script.git {REPO_ROOT}
else:
    print("Repo already present, skipping clone.")

!pip install -q -r {REPO_ROOT}/backend/requirements.txt
!pip install -q pyngrok nest-asyncio

In [ ]:
# Cell 2 — Start the AmicoScript cloud engine
# ⚠️  Run Cell 1 first if you haven't already.
import sys
import os
import asyncio

REPO_ROOT = "/content/amico-script"

if not os.path.exists(os.path.join(REPO_ROOT, "backend", "main.py")):
    raise RuntimeError(
        "Repository not found at " + REPO_ROOT + ".\n"
        "Please run Cell 1 first to clone the repo and install dependencies."
    )

# nest_asyncio must be applied FIRST — Colab already has a running event loop
# and uvicorn would otherwise crash with "asyncio.run() cannot be called
# from a running event loop".
import nest_asyncio
nest_asyncio.apply()

os.chdir(REPO_ROOT)

# Use the absolute path — relative entries in sys.path are not resolved by
# Python's import system inside an IPython kernel.
backend_path = os.path.join(REPO_ROOT, "backend")
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

from pyngrok import ngrok
import uvicorn
from main import app  # resolves correctly now that backend/ is on sys.path

# ── Configure ngrok ──────────────────────────────────────────────────────────
# Paste your token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000).public_url
print("\n" + "=" * 60)
print("✅  AMICOSCRIPT CLOUD ENGINE IS READY")
print(f"\n👉  Copy this URL into AmicoScript › Settings › Colab Bridge URL:\n    {public_url}")
print("=" * 60 + "\n")

os.environ['AMICO_WORD_TIMESTAMPS'] = '1'

# Use uvicorn.Server + await instead of uvicorn.run() — the latter calls
# asyncio.run() which fails inside Colab's already-running event loop.
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()